# Phase 1: Data Acquisition & Quality Checks

**Question of this phase:** *what data do we have, and can we trust it?*

Every coefficient of every later model comes from the rows kept here, so this notebook checks the
data before any modeling.

**The data.** Two public tables from [OpenML](https://www.openml.org/), one accounting year of a
French motor third-party-liability portfolio:

| Table | One row per | Main columns |
|---|---|---|
| `freMTPL2freq` (id 41214) | policy | `Exposure` (years insured), `ClaimNb` (claims reported), rating factors |
| `freMTPL2sev` (id 41215) | claim payment | `IDpol` (policy), `ClaimAmount` (EUR paid) |

**Plan of the notebook:**
1. Load the two tables.
2. Basic checks: duplicates, exposure, claim counts, missing values, claim amounts.
3. Compare the claims reported with the claims paid. This reveals the main data defect of the
   project: a block of 9,387 "policies" that are in fact claims records.
4. Build one clean table, one row per policy, saved in `data/processed/`.

## 0. Setup

In [1]:
import sys  # gives access to Python's list of folders where modules are searched
from pathlib import Path  # file paths that work on every operating system

import numpy as np  # maths on columns of numbers
import pandas as pd  # tables

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # notebooks/ is one level below the project root
sys.path.insert(0, str(PROJECT_ROOT))  # so that "from src... import ..." finds the project's own code

pd.set_option("display.max_columns", 20)  # show up to 20 columns when printing a table
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")  # print decimals as 1,234.5678

## 1. Load the two tables

The first run downloads them from OpenML and saves a copy in `data/raw/`; later runs read the copy.

In [2]:
from src.data.load_data import load_freq_raw, load_sev_raw  # the project's loading functions

freq = load_freq_raw()  # policy table
sev = load_sev_raw()  # payment table

print(f"freMTPL2freq: {freq.shape[0]:,} rows x {freq.shape[1]} columns")  # shape = (rows, columns)
print(f"freMTPL2sev : {sev.shape[0]:,} rows x {sev.shape[1]} columns")

freMTPL2freq: 678,013 rows x 12 columns
freMTPL2sev : 26,639 rows x 2 columns


In [3]:
freq.head()  # the first 5 policies

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1,1,0.1000,D,5,0,55,50,B12,Regular,"1,217.0000",R82
1,3,1,0.7700,D,5,0,55,50,B12,Regular,"1,217.0000",R82
2,5,1,0.7500,B,6,2,52,50,B12,Diesel,54.0000,R22
3,10,1,0.0900,B,7,0,46,50,B12,Diesel,76.0000,R72
4,11,1,0.8400,B,7,0,46,50,B12,Diesel,76.0000,R72


In [4]:
sev.head()  # the first 5 payments

,IDpol,ClaimAmount
0,1552,995.2000
1,1010996,"1,128.1200"
2,4024277,"1,851.1100"
3,4007252,"1,204.0000"
4,4046424,"1,204.0000"


## 2. Basic checks on the policy table

### 2.1 Is each policy listed only once?

In [5]:
n_rows = len(freq)  # number of rows
n_unique_ids = freq["IDpol"].nunique()  # number of different policy numbers
n_duplicated_ids = freq["IDpol"].duplicated().sum()  # rows whose policy number already appeared above

print(f"Rows                          : {n_rows:,}")
print(f"Different policy numbers      : {n_unique_ids:,}")
print(f"Rows with a repeated IDpol    : {n_duplicated_ids:,}")

Rows                          : 678,013
Different policy numbers      : 678,013
Rows with a repeated IDpol    : 0


**Result:** every policy number is unique. Some copies of this dataset found online contain
duplicates; this one does not, so no deduplication is needed.

### 2.2 Exposure (time insured during the year)

`Exposure` is the share of the year the policy was insured: `0.5` = six months. It cannot exceed
one year.

In [6]:
print(freq["Exposure"].describe())  # count, mean, min, quartiles, max

n_above_one_year = (freq["Exposure"] > 1).sum()  # exposures above one year (impossible)
n_zero_or_less = (freq["Exposure"] <= 0).sum()  # exposures equal to zero or negative (impossible)
print(f"\nPolicies with Exposure > 1 year : {n_above_one_year:,} ({n_above_one_year / n_rows:.3%})")
print(f"Policies with Exposure <= 0     : {n_zero_or_less:,}")
print(f"Largest exposure                : {freq['Exposure'].max():.3f} years")

count   678,013.0000
mean          0.5288
std           0.3644
min           0.0027
25%           0.1800
50%           0.4900
75%           0.9900
max           2.0100
Name: Exposure, dtype: float64

Policies with Exposure > 1 year : 1,224 (0.181%)
Policies with Exposure <= 0     : 0
Largest exposure                : 2.010 years


**Problem:** 1,224 policies (0.18%) show more than one year, up to 2.01 years, which is impossible
in a one-year file. **Treatment:** these values are capped at 1.0 when the clean table is built
(Section 5). The policies are kept, because their other information is valid.

### 2.3 Number of claims per policy

In [7]:
print(freq["ClaimNb"].value_counts().sort_index())  # how many policies have 0, 1, 2, ... claims

share_without_claim = (freq["ClaimNb"] == 0).mean()  # share of policies with no claim
print(f"\nShare of policies with no claim : {share_without_claim:.2%}")
print(f"Largest number of claims         : {freq['ClaimNb'].max()}")

ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64

Share of policies with no claim : 94.98%
Largest number of claims         : 16


**Result:** most policies have no claim, as expected in motor insurance; a few have many (up to 16,
plausible for a fleet). The *frequency* (claims per year insured) is computed properly in Phase 2.

### 2.4 Missing values

In [8]:
freq.isnull().sum()  # number of empty cells in each column

IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

**Result:** no missing value.

## 3. Checks on the payment table

In [9]:
print(sev["ClaimAmount"].describe())  # count, mean, min, quartiles, max of the amounts paid
print(f"\nAmounts <= 0   : {(sev['ClaimAmount'] <= 0).sum():,}")
print(f"Missing values : {sev.isnull().sum().sum():,}")
print("\nMost frequent amounts (EUR):")
print(sev["ClaimAmount"].value_counts().head(3))  # the three amounts that appear most often

count      26,639.0000
mean        2,278.5364
std        29,297.4813
min             1.0000
25%           686.8100
50%         1,172.0000
75%         1,228.0800
max     4,075,400.5600
Name: ClaimAmount, dtype: float64

Amounts <= 0   : 0
Missing values : 0

Most frequent amounts (EUR):
ClaimAmount
1,204.0000    4792
1,128.1200    3056
1,172.0000    2072
Name: count, dtype: int64


**Result:** no negative or missing amount. The amounts are very unequal: the median claim is about
EUR 1,172 but the largest is EUR 4.08 million. A few large claims will weigh heavily on any average;
Phase 6 treats them separately. Three amounts (EUR 1,204, 1,128.12 and 1,172) appear thousands of
times: they look like standard settlement amounts.

## 4. Do the claims reported match the claims paid?

`ClaimNb` (policy table) counts the claims **reported**; `freMTPL2sev` lists the claims **paid**.
We count the payments of each policy and compare.

In [10]:
payments_per_policy = sev.groupby("IDpol").agg(  # one row per policy that received payments
    sev_claim_count=("ClaimAmount", "size"),  # number of payments
    sev_claim_amount=("ClaimAmount", "sum"),  # total paid
).reset_index()

merged = freq.merge(payments_per_policy, on="IDpol", how="left")  # attach the payments to each policy
merged["sev_claim_count"] = merged["sev_claim_count"].fillna(0).astype(int)  # no payment -> 0 payment
merged["sev_claim_amount"] = merged["sev_claim_amount"].fillna(0.0)  # no payment -> EUR 0

reported = merged["ClaimNb"].sum()  # claims reported in total
paid = merged["sev_claim_count"].sum()  # claims paid in total
reported_but_unpaid = ((merged["ClaimNb"] > 0) & (merged["sev_claim_count"] == 0)).sum()  # policies with a claim but no payment
orphan_payments = sev.loc[~sev["IDpol"].isin(freq["IDpol"]), "IDpol"].nunique()  # payments whose policy does not exist

print(f"Claims reported                          : {reported:,}")
print(f"Claims paid                              : {paid:,}  ({paid / reported:.1%} of the claims reported)")
print(f"Policies with a claim but no payment     : {reported_but_unpaid:,}")
print(f"Payments with no matching policy (orphans): {orphan_payments}")

Claims reported                          : 36,102


Claims paid                              : 26,444  (73.2% of the claims reported)
Policies with a claim but no payment     : 9,116
Payments with no matching policy (orphans): 6


**Problem:** only about 73% of the claims reported were paid; 9,116 policies have a claim and no
payment. Before deciding what to do, we look at **where** these policies are in the file.

In [11]:
from src.data.load_data import find_claims_only_block_end  # finds the end of the leading block of policies that all have a claim

block_end = find_claims_only_block_end(freq)  # last policy number before the first policy without a claim
in_block = merged["IDpol"] <= block_end  # True for the policies of the leading block

summary_rows = []  # one row per group, built step by step
for label, mask in [(f"IDpol <= {block_end:,}", in_block), (f"IDpol >  {block_end:,}", ~in_block)]:
    part = merged[mask]  # the policies of this group
    claims = part["ClaimNb"].sum()  # claims reported in the group
    exposure = part["Exposure"].clip(upper=1).sum()  # years insured (capped at 1 per policy)
    summary_rows.append({
        "group": label,
        "policies": len(part),
        "share with a claim": (part["ClaimNb"] > 0).mean(),  # share of policies with at least one claim
        "claim frequency": claims / exposure,  # claims per year insured
        "claims paid": part["sev_claim_count"].sum() / claims,  # share of the claims that were paid
        "share of exposure": exposure / merged["Exposure"].clip(upper=1).sum(),  # weight in the portfolio
    })
pd.DataFrame(summary_rows)

,group,policies,share with a claim,claim frequency,claims paid,share of exposure
0,"IDpol <= 24,951",9387,1.0000,2.0712,0.0309,0.0134
1,"IDpol > 24,951",668626,0.0369,0.0740,0.9991,0.9866


**Finding: the unpaid claims are not spread out, they sit in one block.**

| | Leading block (IDpol up to 24,951) | Rest of the file |
|---|---|---|
| Share of policies with a claim | **100%** | 3.7% |
| Claims with a payment | **3.1%** | 99.9% |

In a real portfolio most drivers have no claim in a year. A group in which **every** policy has a
claim is not a group of insured drivers: it is a piece of a claims file pasted at the start of the
policy file.

**Why it matters:** kept, the block would (1) teach the frequency model claims that have no cost,
so the technical premium would be too high, and (2) distort the relativities, because its mix
differs from the portfolio (two thirds Regular fuel against about half elsewhere).

**Treatment:** the block is excluded when the clean table is built. Its end is found from the data
(just before the first policy without a claim, IDpol 24,952), not typed by hand.

## 5. Build the clean table, one row per policy

In [12]:
from src.data.load_data import drop_claims_only_block  # removes the policies up to block_end

policy = drop_claims_only_block(freq, block_end)  # 1. remove the claims-only block
n_block = n_rows - len(policy)  # number of policies removed

policy["Exposure_raw"] = policy["Exposure"]  # 2. keep the original exposure, for traceability
policy["Exposure"] = policy["Exposure"].clip(upper=1.0)  #    then cap it at one year
n_capped = (policy["Exposure_raw"] > 1.0).sum()  #    number of values capped

policy = policy.merge(payments_per_policy, on="IDpol", how="left")  # 3. attach the payments (orphans are dropped here)
policy["sev_claim_count"] = policy["sev_claim_count"].fillna(0).astype(int)  # no payment -> 0
policy["sev_claim_amount"] = policy["sev_claim_amount"].fillna(0.0)  # no payment -> EUR 0

policy["ClaimNb_freq_target"] = policy["ClaimNb"]  # 4. frequency target: claims reported

has_paid_claim = policy["sev_claim_count"] > 0  # 5. severity target: average amount per paid claim
policy["AvgClaimAmount"] = np.where(  #    only defined when there is a payment
    has_paid_claim,
    policy["sev_claim_amount"] / policy["sev_claim_count"].clip(lower=1),  # amount paid / number of payments
    np.nan,  #    NaN = "not applicable" (not a cost of zero)
)
policy["HasPaidClaim"] = has_paid_claim  # True when the policy has at least one paid claim

print(f"Policies excluded (claims-only block)            : {n_block:,}")
print(f"Exposure values capped at 1.0                    : {n_capped:,}")
print(f"Policies with at least one paid claim            : {policy['HasPaidClaim'].sum():,}")
print(f"Policies with a claim but no payment (remaining) : {((policy['ClaimNb'] > 0) & ~policy['HasPaidClaim']).sum():,}")
print(f"Clean table                                      : {policy.shape[0]:,} rows x {policy.shape[1]} columns")

Policies excluded (claims-only block)            : 9,387
Exposure values capped at 1.0                    : 1,224
Policies with at least one paid claim            : 24,651
Policies with a claim but no payment (remaining) : 22
Clean table                                      : 668,626 rows x 18 columns


## 6. Final checks and saving

In [13]:
assert len(policy) == n_rows - n_block  # only the block was removed
assert (policy["IDpol"] > block_end).all()  # no policy of the block is left
assert (policy["Exposure"] <= 1.0).all()  # every exposure is at most one year
assert (policy["ClaimNb_freq_target"] >= 0).all()  # no negative claim count
assert np.isfinite(policy["sev_claim_amount"]).all()  # no infinite or missing amount

out_path = PROJECT_ROOT / "data" / "processed" / "policy_level_clean.parquet"  # where the clean table is saved
out_path.parent.mkdir(parents=True, exist_ok=True)  # create the folder if needed
policy.to_parquet(out_path, index=False)  # save
print(f"Saved: {out_path.relative_to(PROJECT_ROOT).as_posix()}  ({len(policy):,} rows)")

Saved: data/processed/policy_level_clean.parquet  (668,626 rows)


## Phase 1 summary

| Check | Result | Treatment |
|---|---|---|
| Duplicate policy numbers | none | none needed |
| Missing values | none | none needed |
| Exposure above one year | 1,224 policies | capped at 1.0 |
| Amounts <= 0 | none | none needed |
| Claims reported but not paid | 9,116 policies, 9,094 of them in one block | see next row |
| Leading claims-only block (IDpol up to 24,951) | 9,387 policies, all with a claim, 3.1% of claims paid | **excluded**: a claims extract, not insured policies |
| Payments with no policy | 6 policy numbers | dropped |

**Output:** `data/processed/policy_level_clean.parquet`, **668,626 policies**, ready for Phase 2.